In [ ]:
import yaml
import pandas as pd
import os
from collections import defaultdict

def extract_ball_by_ball(filepath):
    with open(filepath, 'r') as f:
        data = yaml.safe_load(f)
    
    # Get the year from the first date in the info section
    match_date = data.get('info', {}).get('dates', ['0000'])[0]
    match_year = str(match_date)[:4] # Extracts '2005' from '2005-06-13'
    
    match_id = os.path.basename(filepath).replace('.yaml', '')
    deliveries = []

    for inning_entry in data.get('innings', []):
        for inning_name, content in inning_entry.items():
            team = content.get('team')
            for delivery in content.get('deliveries', []):
                for ball_num, details in delivery.items():
                    row = {
                        'match_id': match_id,
                        'year': match_year,
                        'innings': inning_name,
                        'batting_team': team,
                        'ball': ball_num,
                        'batsman': details.get('batsman'),
                        'bowler': details.get('bowler'),
                        'runs_batsman': details.get('runs', {}).get('batsman', 0),
                        'runs_extras': details.get('runs', {}).get('extras', 0),
                        'runs_total': details.get('runs', {}).get('total', 0),
                        'is_wicket': 1 if 'wicket' in details else 0,
                        'player_out': details.get('wicket', {}).get('player_out') if 'wicket' in details else None
                    }
                    deliveries.append(row)
    return match_year, deliveries

# MAIN PROCESSING LOOP
folder_path = './your_yaml_folder/'  # Change this to your folder path
year_groups = defaultdict(list)

for filename in os.listdir(folder_path):
    if filename.endswith('.yaml'):
        print(f"Processing {filename}...")
        try:
            year, ball_data = extract_ball_by_ball(os.path.join(folder_path, filename))
            year_groups[year].extend(ball_data)
        except Exception as e:
            print(f"Error in {filename}: {e}")

# Save each year as a separate CSV
for year, data in year_groups.items():
    df = pd.DataFrame(data)
    output_filename = f't20_ball_by_ball_{year}.csv'
    df.to_csv(output_filename, index=False)
    print(f"Saved {output_filename}")